# Credit Card Fraud Detection - Feature Transformation

Applying three methods of feature transformation:

One-hot encoding
* Datetime
* Will characterise as weekend/weekday and day/night, creating two new features.

RFM (recency, frequency, monetary value)
* To Customer ID
* Will keep track of the average spending amount and number of transactions for each customer and for three window sizes. This will lead to the creation of six new features.

Frequency / risk encoding
* To Terminal ID
* To characterise the ‘risk’ of the terminal, taking the average number of frauds that were observed on the terminal for three window sizes. This will lead to the creation of three new features.


In [62]:
import pickle
import os
import datetime
import pandas as pd

In [63]:
# Load the set of pickle files, put them together in a single df, and order by time
def read_from_files(DIR_INPUT, BEGIN_DATE, END_DATE):
    
    files = [os.path.join(DIR_INPUT, f) for f in os.listdir(DIR_INPUT) if f>=BEGIN_DATE+'.pkl' and f<=END_DATE+'.pkl']

    frames = []
    for f in files:
        df = pd.read_pickle(f)
        frames.append(df)
        del df
    df_final = pd.concat(frames)
    
    df_final=df_final.sort_values('TRANSACTION_ID')
    df_final.reset_index(drop=True,inplace=True)
    #  Note: -1 are missing values for real world data 
    df_final=df_final.replace([-1],0)
    
    return df_final

#Save oject as pickle file
def save_object(obj, filename):
    with open(filename, 'wb') as output:
        pickle.dump(obj, output, pickle.HIGHEST_PROTOCOL)

In [64]:
# Loading dataset

DIR_INPUT='../data/simulated-data-raw' 

BEGIN_DATE = "2025-11-01"
END_DATE = "2026-05-02"

print("Load  files")
transactions_df = read_from_files(DIR_INPUT, BEGIN_DATE, END_DATE)
print("{0} transactions loaded, containing {1} fraudulent transactions".format(len(transactions_df),transactions_df.TX_FRAUD.sum()))

Load  files
1754155 transactions loaded, containing 14681 fraudulent transactions


In [65]:
transactions_df.head()

,TRANSACTION_ID,TX_DATETIME,CUSTOMER_ID,TERMINAL_ID,TX_AMOUNT,TX_TIME_SECONDS,TX_TIME_DAYS,TX_FRAUD,TX_FRAUD_SCENARIO
0,0,2025-11-01 00:00:31,596,3156,57.16,31,0,0,0
1,1,2025-11-01 00:02:10,4961,3412,81.51,130,0,0,0
2,2,2025-11-01 00:07:56,2,1365,146.00,476,0,0,0
3,3,2025-11-01 00:09:29,4128,8737,64.49,569,0,0,0
4,4,2025-11-01 00:10:34,927,9906,50.99,634,0,0,0


In [66]:
# Categorises whether transaction is during the weekend (1) or a weekday (0)

def is_weekend(tx_datetime):
    
    # Transform date into weekday (0 is Monday, 6 is Sunday)
    weekday = tx_datetime.weekday()
    # Binary value: 0 if weekday, 1 if weekend
    is_weekend = weekday>=5
    
    return int(is_weekend)

transactions_df['TX_DURING_WEEKEND'] = transactions_df.TX_DATETIME.apply(is_weekend)


In [67]:
# Same for is night (1) or day (0), where day is between 6am - midnight
def is_night(tx_datetime):
    
    # Get the hour of the transaction
    tx_hour = tx_datetime.hour
    # Binary value: 1 if hour less than 6, and 0 otherwise
    is_night = tx_hour<=6
    
    return int(is_night)

%time transactions_df['TX_DURING_NIGHT']=transactions_df.TX_DATETIME.apply(is_night)

CPU times: user 1.02 s, sys: 45.1 ms, total: 1.07 s
Wall time: 1.08 s


In [68]:
# Confirm weekday calculation is correct (nov 2nd was a sunday, nov 3rd was a monday)

transactions_df[transactions_df.TX_TIME_DAYS.isin([1, 2])]

,TRANSACTION_ID,TX_DATETIME,CUSTOMER_ID,TERMINAL_ID,TX_AMOUNT,TX_TIME_SECONDS,TX_TIME_DAYS,TX_FRAUD,TX_FRAUD_SCENARIO,TX_DURING_WEEKEND,TX_DURING_NIGHT
9488,9488,2025-11-02 00:00:11,2221,6047,21.24,86411,1,0,0,1,1
9489,9489,2025-11-02 00:01:08,3535,2848,60.47,86468,1,0,0,1,1
9490,9490,2025-11-02 00:01:16,4974,313,75.04,86476,1,0,0,1,1
9491,9491,2025-11-02 00:01:27,4259,5014,27.93,86487,1,0,0,1,1
9492,9492,2025-11-02 00:01:48,2896,4117,82.06,86508,1,0,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...
28813,28813,2025-11-03 23:54:47,3773,3642,31.75,258887,2,0,0,0,0
28814,28814,2025-11-03 23:55:28,3103,4847,21.97,258928,2,0,0,0,0
28815,28815,2025-11-03 23:56:25,4405,6859,11.08,258985,2,0,0,0,0
28816,28816,2025-11-03 23:57:19,1206,6875,67.38,259039,2,0,0,0,0


In [69]:
# Customer id transformation

def get_customer_spending_behaviour_features(customer_transactions, windows_size_in_days=[1,7,30]):
    
    # Let us first order transactions chronologically
    customer_transactions=customer_transactions.sort_values('TX_DATETIME')
    
    # The transaction date and time is set as the index, which will allow the use of the rolling function 
    customer_transactions.index=customer_transactions.TX_DATETIME
    
    # For each window size
    for window_size in windows_size_in_days:
        
        # Compute the sum of the transaction amounts and the number of transactions for the given window size
        SUM_AMOUNT_TX_WINDOW=customer_transactions['TX_AMOUNT'].rolling(str(window_size)+'d').sum()
        NB_TX_WINDOW=customer_transactions['TX_AMOUNT'].rolling(str(window_size)+'d').count()
    
        # Compute the average transaction amount for the given window size
        # NB_TX_WINDOW is always >0 since current transaction is always included
        AVG_AMOUNT_TX_WINDOW=SUM_AMOUNT_TX_WINDOW/NB_TX_WINDOW
    
        # Save feature values
        customer_transactions['CUSTOMER_ID_NB_TX_'+str(window_size)+'DAY_WINDOW']=list(NB_TX_WINDOW)
        customer_transactions['CUSTOMER_ID_AVG_AMOUNT_'+str(window_size)+'DAY_WINDOW']=list(AVG_AMOUNT_TX_WINDOW)
    
    # Reindex according to transaction IDs
    customer_transactions.index=customer_transactions.TRANSACTION_ID
        
    # And return the dataframe with the new features
    return customer_transactions

In [70]:
spending_behaviour_customer_0=get_customer_spending_behaviour_features(transactions_df[transactions_df.CUSTOMER_ID==0])
spending_behaviour_customer_0

,TRANSACTION_ID,TX_DATETIME,CUSTOMER_ID,TERMINAL_ID,TX_AMOUNT,TX_TIME_SECONDS,TX_TIME_DAYS,TX_FRAUD,TX_FRAUD_SCENARIO,TX_DURING_WEEKEND,TX_DURING_NIGHT,CUSTOMER_ID_NB_TX_1DAY_WINDOW,CUSTOMER_ID_AVG_AMOUNT_1DAY_WINDOW,CUSTOMER_ID_NB_TX_7DAY_WINDOW,CUSTOMER_ID_AVG_AMOUNT_7DAY_WINDOW,CUSTOMER_ID_NB_TX_30DAY_WINDOW,CUSTOMER_ID_AVG_AMOUNT_30DAY_WINDOW
TRANSACTION_ID,,,,,,,,,,,,,,,,,
1758,1758,2025-11-01 07:19:05,0,6076,123.59,26345,0,0,0,1,0,1.0,123.590000,1.0,123.590000,1.0,123.590000
8275,8275,2025-11-01 18:00:16,0,858,77.34,64816,0,0,0,1,0,2.0,100.465000,2.0,100.465000,2.0,100.465000
8640,8640,2025-11-01 19:02:02,0,6698,46.51,68522,0,0,0,1,0,3.0,82.480000,3.0,82.480000,3.0,82.480000
12169,12169,2025-11-02 08:51:06,0,6569,54.72,118266,1,0,0,1,0,3.0,59.523333,4.0,75.540000,4.0,75.540000
15764,15764,2025-11-02 14:05:38,0,7707,63.30,137138,1,0,0,1,0,4.0,60.467500,5.0,73.092000,5.0,73.092000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1750390,1750390,2026-05-02 13:38:41,0,3096,38.23,15773921,182,0,0,1,0,5.0,64.388000,28.0,57.306429,89.0,63.097640
1750758,1750758,2026-05-02 14:10:21,0,9441,43.60,15775821,182,0,0,1,0,6.0,60.923333,29.0,56.833793,89.0,62.433933
1751039,1751039,2026-05-02 14:34:30,0,1138,69.69,15777270,182,0,0,1,0,7.0,62.175714,29.0,57.872414,90.0,62.514556


In [71]:
def make_transactions(group, windows_size_in_days=[1,7,30]):
    row = group.copy()
    row['CUSTOMER_ID'] = group.name
    return get_customer_spending_behaviour_features(row, windows_size_in_days=windows_size_in_days)

transactions_df = (
    transactions_df
    .groupby('CUSTOMER_ID')
    .apply(make_transactions, windows_size_in_days=[1,7,30])
    .reset_index(drop=True)
)
transactions_df

,TRANSACTION_ID,TX_DATETIME,TERMINAL_ID,TX_AMOUNT,TX_TIME_SECONDS,TX_TIME_DAYS,TX_FRAUD,TX_FRAUD_SCENARIO,TX_DURING_WEEKEND,TX_DURING_NIGHT,CUSTOMER_ID,CUSTOMER_ID_NB_TX_1DAY_WINDOW,CUSTOMER_ID_AVG_AMOUNT_1DAY_WINDOW,CUSTOMER_ID_NB_TX_7DAY_WINDOW,CUSTOMER_ID_AVG_AMOUNT_7DAY_WINDOW,CUSTOMER_ID_NB_TX_30DAY_WINDOW,CUSTOMER_ID_AVG_AMOUNT_30DAY_WINDOW
0,1758,2025-11-01 07:19:05,6076,123.59,26345,0,0,0,1,0,0,1.0,123.590000,1.0,123.590000,1.0,123.590000
1,8275,2025-11-01 18:00:16,858,77.34,64816,0,0,0,1,0,0,2.0,100.465000,2.0,100.465000,2.0,100.465000
2,8640,2025-11-01 19:02:02,6698,46.51,68522,0,0,0,1,0,0,3.0,82.480000,3.0,82.480000,3.0,82.480000
3,12169,2025-11-02 08:51:06,6569,54.72,118266,1,0,0,1,0,0,3.0,59.523333,4.0,75.540000,4.0,75.540000
4,15764,2025-11-02 14:05:38,7707,63.30,137138,1,0,0,1,0,0,4.0,60.467500,5.0,73.092000,5.0,73.092000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1754150,1739959,2026-05-01 12:12:38,1804,28.68,15682358,181,0,0,0,0,4999,5.0,25.150000,21.0,36.549524,81.0,41.028642
1754151,1742880,2026-05-01 17:06:15,8677,5.65,15699975,181,0,0,0,0,4999,5.0,21.544000,21.0,34.468095,82.0,40.597195
1754152,1745242,2026-05-02 04:32:41,6500,52.18,15741161,182,0,0,1,1,4999,5.0,29.068000,19.0,35.768947,82.0,40.711707
1754153,1746723,2026-05-02 08:07:27,2251,14.96,15754047,182,0,0,1,0,4999,4.0,25.367500,19.0,34.023158,83.0,40.401446


In [72]:
# Terminal id transformation - the risk score will be defined as the average number of fraudulent transactions 
# that occurred on a terminal ID over a time window. A delay will be added to the windows to account for the fact
# that fraudulent transactions are only detected after an investigation or complaint

In [73]:
def get_count_risk_rolling_window(terminal_transactions, delay_period=7, windows_size_in_days=[1,7,30], feature="TERMINAL_ID"):
    
    terminal_transactions=terminal_transactions.sort_values('TX_DATETIME')
    
    terminal_transactions.index=terminal_transactions.TX_DATETIME
    
    NB_FRAUD_DELAY=terminal_transactions['TX_FRAUD'].rolling(str(delay_period)+'d').sum()
    NB_TX_DELAY=terminal_transactions['TX_FRAUD'].rolling(str(delay_period)+'d').count()
    
    for window_size in windows_size_in_days:
    
        NB_FRAUD_DELAY_WINDOW=terminal_transactions['TX_FRAUD'].rolling(str(delay_period+window_size)+'d').sum()
        NB_TX_DELAY_WINDOW=terminal_transactions['TX_FRAUD'].rolling(str(delay_period+window_size)+'d').count()
    
        NB_FRAUD_WINDOW=NB_FRAUD_DELAY_WINDOW-NB_FRAUD_DELAY
        NB_TX_WINDOW=NB_TX_DELAY_WINDOW-NB_TX_DELAY
    
        RISK_WINDOW=NB_FRAUD_WINDOW/NB_TX_WINDOW
        
        terminal_transactions[feature+'_NB_TX_'+str(window_size)+'DAY_WINDOW']=list(NB_TX_WINDOW)
        terminal_transactions[feature+'_RISK_'+str(window_size)+'DAY_WINDOW']=list(RISK_WINDOW)
        
    terminal_transactions.index=terminal_transactions.TRANSACTION_ID
    
    # Replace NA values with 0 (all undefined risk scores where NB_TX_WINDOW is 0) 
    terminal_transactions.fillna(0,inplace=True)
    
    return terminal_transactions

In [74]:
transactions_df[transactions_df.TX_FRAUD==1]

,TRANSACTION_ID,TX_DATETIME,TERMINAL_ID,TX_AMOUNT,TX_TIME_SECONDS,TX_TIME_DAYS,TX_FRAUD,TX_FRAUD_SCENARIO,TX_DURING_WEEKEND,TX_DURING_NIGHT,CUSTOMER_ID,CUSTOMER_ID_NB_TX_1DAY_WINDOW,CUSTOMER_ID_AVG_AMOUNT_1DAY_WINDOW,CUSTOMER_ID_NB_TX_7DAY_WINDOW,CUSTOMER_ID_AVG_AMOUNT_7DAY_WINDOW,CUSTOMER_ID_NB_TX_30DAY_WINDOW,CUSTOMER_ID_AVG_AMOUNT_30DAY_WINDOW
271,1375940,2026-03-24 12:30:51,2352,22.54,12400251,143,1,2,0,0,0,1.0,22.54000,8.0,50.356250,60.0,55.755333
492,261348,2025-11-28 08:15:51,7631,167.95,2362551,27,1,3,0,0,1,3.0,84.87000,28.0,52.495714,109.0,48.218073
496,265242,2025-11-28 13:57:29,5,205.35,2383049,27,1,3,0,0,1,5.0,97.97400,32.0,55.993750,113.0,49.360088
499,268723,2025-11-29 00:43:05,7372,262.80,2421785,28,1,3,1,1,1,8.0,101.56875,32.0,60.838750,116.0,50.865259
500,280680,2025-11-30 08:12:01,2641,76.55,2535121,29,1,3,1,0,1,1.0,76.55000,31.0,62.253226,117.0,51.084786
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1752094,1097362,2026-02-23 11:22:44,6041,60.97,9890564,114,1,2,0,0,4995,2.0,37.50000,17.0,52.707059,74.0,54.601757
1752919,861561,2026-01-29 17:28:33,4876,17.69,7752513,89,1,2,0,0,4997,2.0,25.50500,8.0,60.757500,61.0,61.717213
1753136,50116,2025-11-06 07:51:57,9536,226.88,460317,5,1,1,0,0,4998,4.0,113.18750,22.0,101.688636,22.0,101.688636
1753137,50212,2025-11-06 08:02:34,1999,248.12,460954,5,1,1,0,0,4998,5.0,140.17400,23.0,108.055217,23.0,108.055217


In [75]:
# Computing these features for the first terminal id containing at least one fraud:

# Get the first terminal ID that contains frauds
transactions_df[transactions_df.TX_FRAUD==0].TERMINAL_ID[0]

np.int64(6076)

In [76]:
get_count_risk_rolling_window(transactions_df[transactions_df.TERMINAL_ID==6076], delay_period=7, windows_size_in_days=[1,7,30])


,TRANSACTION_ID,TX_DATETIME,TERMINAL_ID,TX_AMOUNT,TX_TIME_SECONDS,TX_TIME_DAYS,TX_FRAUD,TX_FRAUD_SCENARIO,TX_DURING_WEEKEND,TX_DURING_NIGHT,...,CUSTOMER_ID_NB_TX_7DAY_WINDOW,CUSTOMER_ID_AVG_AMOUNT_7DAY_WINDOW,CUSTOMER_ID_NB_TX_30DAY_WINDOW,CUSTOMER_ID_AVG_AMOUNT_30DAY_WINDOW,TERMINAL_ID_NB_TX_1DAY_WINDOW,TERMINAL_ID_RISK_1DAY_WINDOW,TERMINAL_ID_NB_TX_7DAY_WINDOW,TERMINAL_ID_RISK_7DAY_WINDOW,TERMINAL_ID_NB_TX_30DAY_WINDOW,TERMINAL_ID_RISK_30DAY_WINDOW
TRANSACTION_ID,,,,,,,,,,,,,,,,,,,,,
1758,1758,2025-11-01 07:19:05,6076,123.59,26345,0,0,0,1,0,...,1.0,123.590000,1.0,123.590000,0.0,0.0,0.0,0.0,0.0,0.0
4124,4124,2025-11-01 11:08:39,6076,71.85,40119,0,0,0,1,0,...,3.0,86.713333,3.0,86.713333,0.0,0.0,0.0,0.0,0.0,0.0
8331,8331,2025-11-01 18:07:36,6076,115.41,65256,0,0,0,1,0,...,3.0,89.003333,3.0,89.003333,0.0,0.0,0.0,0.0,0.0,0.0
26679,26679,2025-11-03 16:01:19,6076,71.94,230479,2,0,0,0,0,...,10.0,62.998000,10.0,62.998000,0.0,0.0,0.0,0.0,0.0,0.0
41069,41069,2025-11-05 08:46:11,6076,85.92,377171,4,0,0,0,0,...,15.0,84.902000,15.0,84.902000,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1733743,1733743,2026-04-30 17:52:06,6076,54.67,15616326,180,0,0,0,0,...,22.0,81.726818,85.0,92.999294,1.0,0.0,7.0,0.0,21.0,0.0
1742412,1742412,2026-05-01 16:05:18,6076,63.72,15696318,181,0,0,0,0,...,17.0,93.962941,67.0,93.785373,0.0,0.0,7.0,0.0,21.0,0.0
1742552,1742552,2026-05-01 16:24:01,6076,25.52,15697441,181,0,0,0,0,...,19.0,30.736842,61.0,33.294918,0.0,0.0,7.0,0.0,21.0,0.0


In [77]:
def get_risk(group, delay_period=7, windows_size_in_days=[1,7,30], feature="TERMINAL_ID"):
    row = group.copy()
    row['TERMINAL_ID'] = group.name
    return get_count_risk_rolling_window(row, delay_period=delay_period, windows_size_in_days=windows_size_in_days, feature=feature)

transactions_df = (
    transactions_df
    .groupby('TERMINAL_ID')
    .apply(get_risk, delay_period=7, windows_size_in_days=[1,7,30], feature="TERMINAL_ID")
)
transactions_df=transactions_df.sort_values('TX_DATETIME').reset_index(drop=True)
transactions_df

,TRANSACTION_ID,TX_DATETIME,TX_AMOUNT,TX_TIME_SECONDS,TX_TIME_DAYS,TX_FRAUD,TX_FRAUD_SCENARIO,TX_DURING_WEEKEND,TX_DURING_NIGHT,CUSTOMER_ID,...,CUSTOMER_ID_AVG_AMOUNT_7DAY_WINDOW,CUSTOMER_ID_NB_TX_30DAY_WINDOW,CUSTOMER_ID_AVG_AMOUNT_30DAY_WINDOW,TERMINAL_ID,TERMINAL_ID_NB_TX_1DAY_WINDOW,TERMINAL_ID_RISK_1DAY_WINDOW,TERMINAL_ID_NB_TX_7DAY_WINDOW,TERMINAL_ID_RISK_7DAY_WINDOW,TERMINAL_ID_NB_TX_30DAY_WINDOW,TERMINAL_ID_RISK_30DAY_WINDOW
0,0,2025-11-01 00:00:31,57.16,31,0,0,0,1,1,596,...,57.160000,1.0,57.160000,3156,0.0,0.0,0.0,0.0,0.0,0.00000
1,1,2025-11-01 00:02:10,81.51,130,0,0,0,1,1,4961,...,81.510000,1.0,81.510000,3412,0.0,0.0,0.0,0.0,0.0,0.00000
2,2,2025-11-01 00:07:56,146.00,476,0,0,0,1,1,2,...,146.000000,1.0,146.000000,1365,0.0,0.0,0.0,0.0,0.0,0.00000
3,3,2025-11-01 00:09:29,64.49,569,0,0,0,1,1,4128,...,64.490000,1.0,64.490000,8737,0.0,0.0,0.0,0.0,0.0,0.00000
4,4,2025-11-01 00:10:34,50.99,634,0,0,0,1,1,927,...,50.990000,1.0,50.990000,9906,0.0,0.0,0.0,0.0,0.0,0.00000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1754150,1754150,2026-05-02 23:56:36,54.24,15810996,182,0,0,1,0,161,...,67.047500,72.0,69.521111,655,1.0,0.0,4.0,0.0,28.0,0.00000
1754151,1754151,2026-05-02 23:57:38,1.23,15811058,182,0,0,1,0,4342,...,22.173810,93.0,24.780753,6181,1.0,0.0,9.0,0.0,39.0,0.00000
1754152,1754152,2026-05-02 23:58:21,6.62,15811101,182,0,0,1,0,618,...,7.400476,65.0,7.864462,1502,1.0,0.0,5.0,0.0,33.0,0.00000
1754153,1754153,2026-05-02 23:59:52,55.40,15811192,182,0,0,1,0,4056,...,107.052500,51.0,102.919608,3067,1.0,0.0,6.0,0.0,28.0,0.00000


In [79]:
# Saving the dataset

DIR_OUTPUT = "../data/simulated-data-transformed/"

if not os.path.exists(DIR_OUTPUT):
    os.makedirs(DIR_OUTPUT)

start_date = datetime.datetime.strptime("2025-11-01", "%Y-%m-%d")

for day in range(transactions_df.TX_TIME_DAYS.max()+1):
    
    transactions_day = transactions_df[transactions_df.TX_TIME_DAYS==day].sort_values('TX_TIME_SECONDS')
    
    date = start_date + datetime.timedelta(days=day)
    filename_output = date.strftime("%Y-%m-%d")+'.pkl'
    
    transactions_day.to_pickle(DIR_OUTPUT+filename_output, protocol=4)